## Compare features selection methods

In [ ]:
import pandas as pd
from pathlib import Path

results_root = Path("../saved_data/results")
cohort_list = [l.strip() for l in open("../saved_data/cohorts/DTB/representative_cohorts.txt").readlines()[1:] if l.strip()]

def load_result(cohort, prefix, fold=0):
    path = (results_root / cohort / "random_forest" / prefix / f"fold_{fold}"
            / "agg_int_24" / "impute_fill" / "variant_VMD" / "results_final.csv")
    if not path.exists():
        return None
    df = pd.read_csv(path, on_bad_lines="skip")
    test_rows = df[df["split"] == "test"]
    if test_rows.empty:
        return None
    row = test_rows.iloc[-1]  # last/final run for that split
    return {"auroc": row["auroc"], "auprc": row["auprc"], "f1": row["f1"]}

rows = []
for cohort in cohort_list:
    # old run: prefer the 030626 rerun where it exists, else fall back to the original 200526/190526 run
    old_prefix = next(
        (c for c in ["030626", "200526", "190526"]
         if (results_root / cohort / "random_forest" / c / "fold_0").exists()),
        None,
    )

    entry = {"cohort": cohort}
    for label, prefix in [("old", old_prefix), ("top100", "corr_feat_training"), ("mrmr", "mrmr_feat_training")]:
        res = load_result(cohort, prefix) if prefix else None
        entry[f"{label}_auroc"] = res["auroc"] if res else None
        entry[f"{label}_auprc"] = res["auprc"] if res else None
        entry[f"{label}_f1"] = res["f1"] if res else None
    rows.append(entry)

results_df = pd.DataFrame(rows)
#print("Missing values per column:")
#print(results_df.isna().sum())
#results_df
results_df= results_df.dropna(subset=["old_auroc"])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

variants = ["old_auroc", "top100_auroc", "mrmr_auroc"]
labels = ["mimic-top100", "corr-top100", "MRMR"]
colors = ["#2a78d6", "#008300", "#e87ba4"]

data = [results_df[v].dropna().values for v in variants]

fig, ax = plt.subplots(figsize=(7, 5))

bp = ax.boxplot(
    data,
    labels=labels,
    widths=0.5,
    patch_artist=True,
    showfliers=False,
    medianprops=dict(color="black", linewidth=2),
)

for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.25)
    patch.set_edgecolor(color)
    patch.set_linewidth(2)

for i, (vals, color) in enumerate(zip(data, colors), start=1):
    jitter = np.random.normal(loc=i, scale=0.06, size=len(vals))
    ax.scatter(jitter, vals, color=color, alpha=0.6, s=18, zorder=3)

ax.set_ylabel("AUROC")
ax.set_title("Random Forest AUROC by feature selection (100 cohorts, fold 0)")
ax.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
from scipy import stats
from statsmodels.stats.multitest import multipletests
import pandas as pd

pairs = [
    ("mrmr_auroc", "top100_auroc"),
    ("top100_auroc", "old_auroc"),
    ("mrmr_auroc", "old_auroc"),
]

rows = []
for a, b in pairs:
    sub = results_df[[a, b]].dropna()
    t_stat, t_p = stats.ttest_rel(sub[a], sub[b])
    w_stat, w_p = stats.wilcoxon(sub[a], sub[b])
    rows.append({
        "comparison": f"{a} vs {b}",
        "n": len(sub),
        "mean_diff": (sub[a] - sub[b]).mean(),
        "ttest_p": t_p,
        "wilcoxon_p": w_p,
    })

sig_df = pd.DataFrame(rows)

# Holm-Bonferroni correction across the 3 comparisons, for each test separately
for col in ["ttest_p", "wilcoxon_p"]:
    reject, p_corrected, _, _ = multipletests(sig_df[col], method="holm")
    sig_df[f"{col}_corrected"] = p_corrected
    sig_df[f"{col}_significant"] = reject

sig_df


In [ ]:
results_df.sort_values(by= "top100_auroc",ascending=False)

In [ ]:
results_df[results_df["cohort"].str.startswith("J")]


In [ ]:
import pandas as pd
from pathlib import Path
import re

cohort = "J38-J06"  # set the cohort you want

prefixes = {
    "old": next((p for p in ["030626", "200526", "190526"]
                 if (Path("../saved_data/results")/cohort/"random_forest"/p/"fold_0").exists()), None),
    "top100": "corr_feat_training",
    "mrmr": "mrmr_feat_training",
}

top10_by_variant = {}
for label, prefix in prefixes.items():
    if prefix is None:
        continue
    fi_path = (Path("../saved_data/results") / cohort / "random_forest" / prefix
               / "fold_0" / "agg_int_24" / "impute_fill" / "variant_VMD" / "feature_importances.csv")
    fi = pd.read_csv(fi_path)
    top10_by_variant[label] = fi.sort_values("importance", ascending=False).head(10).reset_index(drop=True)

'''for label, df in top10_by_variant.items():
    print(f"\n=== {label} ({cohort}) ===")
    print(df)'''


def label_features(df):
    df = df.copy()
    parsed = df["feature"].str.extract(r"^(\d+)(_.*)?$")
    df["itemid"] = pd.to_numeric(parsed[0], errors="coerce").astype("Int64")
    df["suffix"] = parsed[1].fillna("")
    df = df.merge(mimic_d_items[["itemid", "label"]], on="itemid", how="left")
    return df

top10_labeled = {label: label_features(df) for label, df in top10_by_variant.items()}

for label, df in top10_labeled.items():
    print(f"\n=== {label} ===")
    print(df[["feature", "itemid", "suffix", "label", "importance"]])


In [ ]:
comparison = pd.DataFrame({
    label: (df["label"].fillna(df["feature"]) + " (" + df["feature"] + ")").values
    for label, df in top10_labeled.items()
})
comparison = comparison.rename(columns={
    "old": "mimic-top100",
    "top100": "corr-top100",
    "mrmr": "MRMR",
})
comparison

comparison.index = range(1, 11)
comparison.index.name = "rank"
comparison


In [ ]:
import pickle
with open ("../data/top_features/mimic_top100_features.pkl", "rb") as f:
    mimic_top_features = pickle.load(f)

In [ ]:
check_ids = ["50815","50804"]
{i: (i in mimic_top_features) for i in check_ids}


In [ ]:
sub_mrmr = results_df[["cohort", "mrmr_auroc", "old_auroc"]].dropna()
sub_top100 = results_df[["cohort", "top100_auroc", "old_auroc"]].dropna()

mrmr_better = set(sub_mrmr.loc[sub_mrmr["mrmr_auroc"] > sub_mrmr["old_auroc"], "cohort"])
top100_better = set(sub_top100.loc[sub_top100["top100_auroc"] > sub_top100["old_auroc"], "cohort"])

print(f"mrmr better than old: {len(mrmr_better)} cohorts")
print(f"top100 better than old: {len(top100_better)} cohorts")
print(f"overlap (both better): {len(mrmr_better & top100_better)}")
print(f"only mrmr better: {len(mrmr_better - top100_better)}")
print(f"only top100 better: {len(top100_better - mrmr_better)}")
print(f"neither better: {len(set(results_df['cohort']) - mrmr_better - top100_better)}")

results_df[results_df["cohort"].isin(mrmr_better - top100_better)][["cohort", "old_auroc", "top100_auroc", "mrmr_auroc"]]
